In [2]:
from langgraph.graph import  StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [3]:
load_dotenv()

True

In [4]:
llm = ChatGroq(model= "llama-3.1-8b-instant")

In [5]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [6]:
def generate_joke(state: JokeState):
    prompt= f'generate a joke on the topic {state['topic']}'
    response= llm.invoke(prompt).content
    return {'joke': response}

In [7]:
def generate_explanation(state: JokeState):
    prompt= f'write an explanation for the joke {state['joke']}'
    response= llm.invoke(prompt).content
    return {'explanation': response}

In [8]:
graph= StateGraph(JokeState)
graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explanation',generate_explanation)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explanation')
graph.add_edge('generate_explanation',END)

checkpointer= InMemorySaver()
workflow= graph.compile(checkpointer=checkpointer)

In [9]:
config1= {'configurable': {'thread_id':'1'}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza go to the doctor? \n\nBecause it was feeling a little crusty.',
 'explanation': 'The joke "Why did the pizza go to the doctor? Because it was feeling a little crusty" is a play on words that uses a pun to create humor. \n\nIn this joke, the word "crusty" has a double meaning. On one hand, it refers to the crusty texture of a pizza, which is a characteristic of the food. On the other hand, "crusty" is also an idiomatic expression that means being irritable or having a rough temperament, similar to being a little "grumpy."\n\nThe joke relies on this wordplay to create a clever and unexpected twist. The setup ("Why did the pizza go to the doctor?") primes the listener to expect a reason related to the pizza\'s physical condition or a health issue. However, the punchline ("Because it was feeling a little crusty") subverts this expectation by using the word "crusty" in its secondary meaning, creating a humorous connection between the pizza\'s t

In [10]:
config2={'configurable': {'thread_id': '2'}}
workflow.invoke({'topic': 'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti go to therapy? \n\nBecause it was feeling a little "twisted" and wanted to get to the "root" of its problems.',
 'explanation': 'The joke "Why did the spaghetti go to therapy?" is a play on words. It\'s a pun, which is a form of wordplay that exploits multiple meanings of words or phrases.\n\nThe joke relies on two layers of meaning:\n\n1. The first layer is the literal meaning: spaghetti is a type of food, and going to therapy implies that the spaghetti has some kind of emotional or psychological issue.\n2. The second layer is the pun: "twisted" has a double meaning. In the context of spaghetti, "twisted" refers to the twisted shape of the noodles. However, in the context of mental health, "twisted" can also imply being emotionally disturbed or having a distorted view of reality.\n3. The third layer is the pun on "root": In the context of spaghetti, "root" is a clever play on words, referencing the root vegetable, but also referring t